# DORAnet → enzyme hypotheses → DNA design

This notebook converts DORAnet-generated reaction networks into concise downstream design tables: parsed reactions, enzyme hypotheses, enzyme-candidate templates, and non-operational DNA design plans for expert review.

In [1]:
from pathlib import Path
import glob
import json
import os
import re
import textwrap
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem
import time
import requests
from io import StringIO

/users/sghosh6/.conda/envs/doranet_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [35]:
from DORA_XGB import DORA_XGB
by_desc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_descending_MW')
by_asc_MW_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'by_ascending_MW')
add_concat_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_concat')
add_subtract_model = DORA_XGB.feasibility_classifier(cofactor_positioning = 'add_subtract')

/users/sghosh6/.conda/envs/doranet_env/lib/python3.10/site-packages/xgboost/compat.py:105: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
resultsDir = os.path.join(dataDir, 'Results/combinedEbolaVirus_bestMACAW_allDB_generative/')
DORANETmoleculesDataDir = os.path.join(resultsDir, 'DORAnet/InoAdeGuoXan/PATHWAY_reconstruction/')
DNADesignResultsDir = os.path.join(resultsDir, 'DNA_design/')
os.makedirs(DNADesignResultsDir, exist_ok=True)

print("Reading DORAnet results from:", DORANETmoleculesDataDir)
print("DNA design results will be saved at:", DNADesignResultsDir)

Reading DORAnet results from: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/PATHWAY_reconstruction/
DNA design results will be saved at: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/


In [3]:
# Subdirectories inside DNA_design
reactionResultsDir = os.path.join(DNADesignResultsDir, "01_reactions/")
moleculeResultsDir = os.path.join(DNADesignResultsDir, "02_molecules/")
enzymeMappingResultsDir = os.path.join(DNADesignResultsDir, "03_enzyme_mapping/")
routeResultsDir    = os.path.join(DNADesignResultsDir, "04_routes/")
sequenceResultsDir = os.path.join(DNADesignResultsDir, "05_uniprot_sequences/")
dnaResultsDir      = os.path.join(DNADesignResultsDir, "06_optimized_dna/")
handoffResultsDir  = os.path.join(DNADesignResultsDir, "07_webtool_handoff/")

# Create all directories
for dirPath in [
    DNADesignResultsDir,
    reactionResultsDir,
    moleculeResultsDir,
    DNADesignResultsDir,
    routeResultsDir,
    sequenceResultsDir,
    dnaResultsDir,
    handoffResultsDir,
]:
    os.makedirs(dirPath, exist_ok=True)

print("reactionResultsDir  :", reactionResultsDir)

reactionResultsDir  : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/01_reactions/


## 1. Discover DORAnet JSON files

In [4]:
DORANETmoleculesDataDir = Path(DORANETmoleculesDataDir)  

def extractFirstInt(text):
    match = re.search(r"\d+", text)
    return int(match.group(0)) if match else -1

# Find all JSON files anywhere under root
allJsonPaths = sorted(DORANETmoleculesDataDir.rglob("*.json"), key=lambda p: str(p).lower())

fileInfoList = []
for jsonPath in allJsonPaths:
    parentDir = jsonPath.parent
    fileInfoList.append({
        "dirName": parentDir.name,
        "dirPath": str(parentDir),
        "jsonName": jsonPath.name,
        "jsonPath": str(jsonPath),
        "folderNum": extractFirstInt(parentDir.name),  # optional helper field
    })

print(f"Root directory        : {DORANETmoleculesDataDir}")
print(f"JSON files found      : {len(fileInfoList)}")
print(f"Unique folders w/JSON : {len({x['dirPath'] for x in fileInfoList})}")

Root directory        : /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DORAnet/InoAdeGuoXan/PATHWAY_reconstruction
JSON files found      : 20
Unique folders w/JSON : 20


## 2. Parse DORAnet reaction strings

In [5]:
def splitMoleculeString(moleculeString):
    return [mol for mol in str(moleculeString).split(".") if mol]

def parseDoranetReaction(rxnString, fileInfo):
    parts = str(rxnString).split(">")
    if len(parts) != 4:
        raise ValueError(f"Expected 4 fields separated by '>'; found {len(parts)}")

    reactants, ruleName, metaBlock, products = parts
    metaParts = (metaBlock.split("$") + [None, None, None, None])[:4]
    thermo, reactantStoich, productStoich, reactionType = metaParts

    return {
        "reactants": reactants,
        "products": products,
        "reactionString": f"{reactants} >> {products}",
        "ruleName": ruleName,
        "thermo": thermo,
        "reactantStoich": reactantStoich,
        "productStoich": productStoich,
        "reactionType": reactionType,
        "numReactantMolecules": len(splitMoleculeString(reactants)),
        "numProductMolecules": len(splitMoleculeString(products)),
        "sourceFolderNum": fileInfo.get("folderNum", -1),
        "sourceDirectory": fileInfo.get("dirName", ""),
        "sourceDirectoryPath": fileInfo.get("dirPath", ""),
        "sourceJsonName": fileInfo.get("jsonName", ""),
        "sourceJsonPath": fileInfo.get("jsonPath", ""),
    }

reactionRecords = []
jsonReadErrors = []
uniqueReactantMolecules = set()
uniqueProductMolecules = set()

for fileInfo in tqdm(fileInfoList, desc="Reading DORAnet JSON"):
    try:
        with open(fileInfo["jsonPath"], "r", encoding="utf-8") as f:
            reactionList = json.load(f)

        # handle both list and dict payloads
        if isinstance(reactionList, dict):
            # try common key names; otherwise fail clearly
            for k in ["reactions", "reactionList", "data"]:
                if k in reactionList and isinstance(reactionList[k], list):
                    reactionList = reactionList[k]
                    break
            else:
                raise ValueError("JSON is dict but no reaction list key found")

        if not isinstance(reactionList, list):
            raise ValueError(f"Expected list of reaction strings, got {type(reactionList)}")

        for rxnString in reactionList:
            record = parseDoranetReaction(rxnString, fileInfo)
            reactionRecords.append(record)
            uniqueReactantMolecules.update(splitMoleculeString(record["reactants"]))
            uniqueProductMolecules.update(splitMoleculeString(record["products"]))

    except Exception as exc:
        jsonReadErrors.append({**fileInfo, "error": str(exc)})

if not reactionRecords:
    raise RuntimeError("No reactions loaded. Check JSON discovery and reaction format.")

reactionDF = pd.DataFrame(reactionRecords).reset_index(drop=True)

# safer path handling
outputPath = Path(DNADesignResultsDir) / "reactionDF.csv"
reactionDF.to_csv(outputPath, index=False)

print(f"Reactions loaded     : {len(reactionDF):,}")
print(f"Failed JSON files    : {len(jsonReadErrors):,}")
print(f"Unique reactant mols : {len(uniqueReactantMolecules):,}")
print(f"Unique product mols  : {len(uniqueProductMolecules):,}")
print(f"Saved: {outputPath}")

reactionDF.head()

Reading DORAnet JSON: 100%|████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 24.21it/s]


Reactions loaded     : 176,400
Failed JSON files    : 0
Unique reactant mols : 283
Unique product mols  : 4,730
Saved: /mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/reactionDF.csv


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,sourceFolderNum,sourceDirectory,sourceDirectoryPath,sourceJsonName,sourceJsonPath
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...


### Keep only enzymatic DORAnet reactions

In [6]:
enzymaticReactionDF = reactionDF[
    reactionDF["reactionType"].astype(str).str.lower().str.contains(
        "enzyme|enzymatic|bio|biological",
        na=False
    )
].copy()

print(f"Total reactions: {len(reactionDF):,}")
print(f"Likely enzymatic reactions: {len(enzymaticReactionDF):,}")

enzymaticReactionDF.head()

Total reactions: 176,400
Likely enzymatic reactions: 176,400


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,sourceFolderNum,sourceDirectory,sourceDirectoryPath,sourceJsonName,sourceJsonPath
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,high_pPotency_molecule_pathway1,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,high_pPotency_molecule_pathway1_network_pretre...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...


### Use `DORA-XGB` to get feasibility score

In [40]:
reactionDF_DORAXGB = enzymaticReactionDF.copy()
#reactionDF_DORAXGB = enzymaticReactionDF.head(50).copy()

# Clean reaction string for model input
reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

def getFeasibilityScoresAndLabels(rxnStr):
    return pd.Series({
        "feasibilityScore_rule1": by_desc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule1": by_desc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule2": by_asc_MW_model.predict_proba(rxnStr),
        "feasibilityLabel_rule2": by_asc_MW_model.predict_label(rxnStr),

        "feasibilityScore_rule3": add_concat_model.predict_proba(rxnStr),
        "feasibilityLabel_rule3": add_concat_model.predict_label(rxnStr),

        "feasibilityScore_rule4": add_subtract_model.predict_proba(rxnStr),
        "feasibilityLabel_rule4": add_subtract_model.predict_label(rxnStr),
    })


reactionDF_DORAXGB["rxn_str"] = (
    reactionDF_DORAXGB["reactionString"]
    .astype(str)
    .str.replace(" ", "", regex=False)
)

reactionDF_DORAXGB[
    [
        "feasibilityScore_rule1",
        "feasibilityLabel_rule1",
        "feasibilityScore_rule2",
        "feasibilityLabel_rule2",
        "feasibilityScore_rule3",
        "feasibilityLabel_rule3",
        "feasibilityScore_rule4",
        "feasibilityLabel_rule4",
    ]
] = reactionDF_DORAXGB["rxn_str"].apply(getFeasibilityScoresAndLabels)

reactionDF_DORAXGB

,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sourceJsonPath,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,0.000647,0.0,0.884856,1.0,0.729129,1.0,0.664541,1.0
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,0.000875,0.0,0.020655,0.0,0.001980,0.0,0.000030,0.0
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,0.477059,0.0,0.012712,0.0,0.327328,0.0,0.049994,0.0
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,0.191322,0.0,0.002167,0.0,0.000062,0.0,0.784046,1.0
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,0.506250,0.0,0.413190,0.0,0.227389,0.0,0.278599,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176395,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,COC(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H]...,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,rule0003_177,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,0.128867,0.0,0.179263,0.0,0.056613,0.0,0.003117,0.0
176396,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,CO[C@@H]1[C@H](OS(=O)(=O)O)[C@@H](C=O)O[C@H]1n...,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,rule0048_5,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,0.018481,0.0,0.958909,1.0,0.637625,1.0,0.646907,0.0
176397,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,C[C@]12O[C@@]1(CO)O[C@@H](n1cnc3c(=O)[nH]cnc31...,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,0.000203,0.0,0.001115,0.0,0.050057,0.0,0.026536,0.0
176398,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)O[C@H]1[C@H]2O[C@]2(n2cnc3c(=O)[nH]cnc32...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,rule0062_19,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,0.000353,0.0,0.291174,0.0,0.023250,0.0,0.017795,0.0


### Keep only `high feasible` reactions

In [43]:
reactionDF_DORAXGB_highFeasibility = (
    reactionDF_DORAXGB[
        (reactionDF_DORAXGB["feasibilityLabel_rule1"] == 1)
        & (reactionDF_DORAXGB["feasibilityScore_rule1"].notna())
    ]
    .sort_values(by="feasibilityScore_rule1", ascending=False)
    .reset_index(drop=True)
)


uniqueReactantStrings_high = set(reactionDF_DORAXGB_highFeasibility["reactants"])
uniqueProductStrings_high = set(reactionDF_DORAXGB_highFeasibility["products"])

uniqueReactantMolecules_high = {
    mol
    for reactants in reactionDF_DORAXGB_highFeasibility["reactants"]
    for mol in str(reactants).split(".")
}

uniqueProductMolecules_high = {
    mol
    for products in reactionDF_DORAXGB_highFeasibility["products"]
    for mol in str(products).split(".")
}

print(f"Number of high-feasibility reactions: {len(reactionDF_DORAXGB_highFeasibility)}")
print(f"Number of unique reactant strings: {len(uniqueReactantStrings_high)}")
print(f"Number of unique product strings: {len(uniqueProductStrings_high)}")
print(f"Number of unique individual reactant molecules: {len(uniqueReactantMolecules_high)}")
print(f"Number of unique individual product molecules: {len(uniqueProductMolecules_high)}")
reactionDF_DORAXGB_highFeasibility.to_csv(os.path.join(DNADesignResultsDir, "DORAXGB_highFeasibility_reactions.csv"),index=False)
reactionDF_DORAXGB_highFeasibility

Number of high-feasibility reactions: 17400
Number of unique reactant strings: 564
Number of unique product strings: 705
Number of unique individual reactant molecules: 241
Number of unique individual product molecules: 640


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sourceJsonPath,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
1,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
2,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
3,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
4,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17395,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17396,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17397,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17398,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0


In [45]:
reactionDF_DORAXGB_highFeasibility = pd.read_csv(os.path.join(DNADesignResultsDir, "DORAXGB_highFeasibility_reactions.csv"))
reactionDF_DORAXGB_highFeasibility

,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,sourceJsonPath,rxn_str,feasibilityScore_rule1,feasibilityLabel_rule1,feasibilityScore_rule2,feasibilityLabel_rule2,feasibilityScore_rule3,feasibilityLabel_rule3,feasibilityScore_rule4,feasibilityLabel_rule4
0,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
1,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
2,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
3,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
4,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,0.999332,1.0,0.979676,1.0,0.316587,0.0,0.438167,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17395,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17396,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17397,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0
17398,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,CC(C)(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,rule0043_12,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,/mnt/data.ese/nfs/users/sghosh6/DTRA_project/M...,CC(O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@@...,0.583536,1.0,0.935283,1.0,0.063748,0.0,0.192906,0.0


### Summarize unique DORAnet rules

Many reactions may use the same rule. We do not want to annotate millions of reactions one by one. First annotate the rules.

In [47]:
enzymaticReactionDF = reactionDF_DORAXGB_highFeasibility.copy()
ruleSummaryDF = (
    enzymaticReactionDF
    .groupby(["ruleName", "reactionType"], dropna=False)
    .agg(
        numReactions=("reactionString", "count"),
        numStarterSources=("sourceFolderNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("reactants", "first"),
        exampleProducts=("products", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

print(f"unique DORAnet rules    : {enzymaticReactionDF['ruleName'].nunique():,}")
ruleSummaryDF

unique DORAnet rules    : 68


,ruleName,reactionType,numReactions,numStarterSources,exampleReaction,exampleReactants,exampleProducts
22,rule0011_51,Enzymatic,3460,20,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C)nc32...
21,rule0011_50,Enzymatic,2360,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...
42,rule0043_12,Enzymatic,2020,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,CC(C(=O)O)C(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]...
52,rule0121_1,Enzymatic,1520,20,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...
67,rule0491_2,Enzymatic,1380,20,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H]...
...,...,...,...,...,...,...,...
46,rule0062_17,Enzymatic,20,20,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)O.CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)...,CC(=O)OC(C)=O.CC(C)(COP(=O)(O)OP(=O)(O)OC[C@H]...
15,rule0009_40,Enzymatic,20,20,CC(=O)O.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,CC(=O)O.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)...,CC(=O)O[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1...
16,rule0009_41,Enzymatic,20,20,CO.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O[...,CO.O=c1ccn([C@@H]2O[C@H](COP(=O)(O)OP(=O)(O)O[...,CO[C@H]1O[C@H](CO)[C@@H](O)[C@H](O)[C@H]1O.O=c...
65,rule0394_7,Enzymatic,20,20,N#CC(O)(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,N#CC(O)(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...,C#N.O=C(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[...


### Add required placeholder columns

In [48]:
enzymeAnnotationDF = ruleSummaryDF.copy()

enzymeAnnotationDF["suggestedEnzymeClass"] = ""
enzymeAnnotationDF["ecNumber"] = ""
enzymeAnnotationDF["enzymeName"] = ""
enzymeAnnotationDF["uniprotAccession"] = ""
enzymeAnnotationDF["sourceDatabase"] = ""
enzymeAnnotationDF["reactionSimilarity"] = ""
enzymeAnnotationDF["enzymeConfidence"] = ""
enzymeAnnotationDF["notes"] = ""
enzymeAnnotationDF.head()

,ruleName,reactionType,numReactions,numStarterSources,exampleReaction,exampleReactants,exampleProducts,suggestedEnzymeClass,ecNumber,enzymeName,uniprotAccession,sourceDatabase,reactionSimilarity,enzymeConfidence,notes
22,rule0011_51,Enzymatic,3460,20,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]c(C)nc32)...,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]c(C)nc32...,,,,,,,,
21,rule0011_50,Enzymatic,2360,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,COC[C@H]1O[C@@H](n2c[nH+]c3c(=O)[nH]cnc32)[C@H...,,,,,,,,
42,rule0043_12,Enzymatic,2020,20,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,C[S+](CC[C@H](N)C(=O)O)C[C@H]1O[C@@H](n2cnc3c(...,CC(C(=O)O)C(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]...,,,,,,,,
52,rule0121_1,Enzymatic,1520,20,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,CO[C@@H]1[C@H](O)[C@@H](COP(=O)(O)O)O[C@H]1n1c...,,,,,,,,
67,rule0491_2,Enzymatic,1380,20,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(OC=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H]...,,,,,,,,


## 3. Read DORAnet reaction ruleset file

In [49]:
doranetRulesetDF = pd.read_csv(dataDir + "/DORAnet/JN3604IMT_rules.tsv", sep="\t")

print(doranetRulesetDF.shape)
print(doranetRulesetDF.columns.tolist())
doranetRulesetDF.head()

(3604, 5)
['Name', 'Reactants', 'SMARTS', 'Products', 'Comments']


,Name,Reactants,SMARTS,Products,Comments
0,rule0001_01,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P97259;P97402;Q08834;Q09328;Q16408;Q3V5L5;Q765...
1,rule0001_02,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A0K8D0;A0KHH6;A0KZ10;A0L8R9;A0Q7X9;A1A7M6;A1K6...
2,rule0001_03,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,P0DMP6;P0DMP7;P16442;Q15951;Q1RLK6;Q3USF0;Q5HZ...
3,rule0001_04,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,A4IID1;O19071;O77836;P23336;P26572;P27115;P278...
4,rule0001_05,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,Any;Any,NaN


### Find the number of common reaction rules between `DORAnet` diversification and ruleset table from actual package

In [10]:
reactionRuleSet = set(reactionDF["ruleName"].dropna().astype(str).str.strip())
tsvRuleSet = set(doranetRulesetDF["Name"].dropna().astype(str).str.strip())

matchedRuleSet = reactionRuleSet.intersection(tsvRuleSet)
missingRuleSet = reactionRuleSet.difference(tsvRuleSet)

print(f"Rules present in DORAnet diversification run: {len(reactionRuleSet):,}")
print(f"Rules matched from DORAnet package: {len(matchedRuleSet):,}")
print(f"Rules missing from DORAnet package: {len(missingRuleSet):,}")

print("\nMatched rules:")
print(sorted(list(matchedRuleSet))[:20])

print("\nMissing rules:")
print(sorted(list(missingRuleSet))[:20])

Rules present in DORAnet diversification run: 168
Rules matched from DORAnet package: 168
Rules missing from DORAnet package: 0

Matched rules:
['rule0001_86', 'rule0001_87', 'rule0001_88', 'rule0001_90', 'rule0002_142', 'rule0002_148', 'rule0002_153', 'rule0002_154', 'rule0003_152', 'rule0003_170', 'rule0003_171', 'rule0003_173', 'rule0003_174', 'rule0003_175', 'rule0003_176', 'rule0003_177', 'rule0004_13', 'rule0005_60', 'rule0005_61', 'rule0005_62']

Missing rules:
[]


### Extract `UniProt IDs` from `DORAnet` reaction rules

In [11]:
def splitUniProtIds(idString):
    if pd.isna(idString):
        return []
    return [x.strip() for x in str(idString).split(";") if x.strip()]


ruleInfoDF = doranetRulesetDF.copy()

ruleInfoDF = ruleInfoDF.rename(columns={
    "Name": "ruleName",
    "Reactants": "ruleReactants",
    "SMARTS": "ruleSMARTS",
    "Products": "ruleProducts",
    "Comments": "candidateUniProtRaw",
})

ruleInfoDF["ruleName"] = ruleInfoDF["ruleName"].astype(str).str.strip()
ruleInfoDF["candidateUniProtList"] = ruleInfoDF["candidateUniProtRaw"].apply(splitUniProtIds)
ruleInfoDF["numCandidateUniProt"] = ruleInfoDF["candidateUniProtList"].apply(len)

ruleInfoDF = ruleInfoDF[
    [
        "ruleName",
        "ruleReactants",
        "ruleProducts",
        "ruleSMARTS",
        "candidateUniProtRaw",
        "numCandidateUniProt",
    ]
].copy()



print(f"ruleInfoDF rows: {len(ruleInfoDF):,}")
ruleInfoDF

ruleInfoDF rows: 3,604


,ruleName,ruleReactants,ruleProducts,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt
0,rule0001_01,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,P97259;P97402;Q08834;Q09328;Q16408;Q3V5L5;Q765...,10
1,rule0001_02,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,A0K8D0;A0KHH6;A0KZ10;A0L8R9;A0Q7X9;A1A7M6;A1K6...,263
2,rule0001_03,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,P0DMP6;P0DMP7;P16442;Q15951;Q1RLK6;Q3USF0;Q5HZ...,29
3,rule0001_04,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,A4IID1;O19071;O77836;P23336;P26572;P27115;P278...,42
4,rule0001_05,Any;Any,Any;Any,[#6;$([#6&R]1(-&!@[#8&!R]-&!@[#15&!R](=&!@[#8&...,NaN,0
...,...,...,...,...,...,...
3599,rule1152_1,Any;Any,Any;Any,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,Q8ZUN8,1
3600,rule1152_2,Any;Any,Any;Any,[#6;!$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[...,B9TTF1,1
3601,rule1164_1,Any;WATER,Any;H2O2,[#6;$([#6&!R]-&!@[#6&R]1:&@[#6&R]:&@[#6&R]:&@[...,Q972I2,1
3602,rule1165_1,Any;H2O2,Any;WATER,[#6;$([#6&!R]-[#6&R]1:&@[#6&R]:&@[#6&R]:&@[#6&...,Q972I2,1


## 4. Merge rule information into `reactionDF`

This connects DORAnet reactions with the rule SMARTS and UniProt candidate list.

In [12]:
reactionWithRuleDF = reactionDF.merge(ruleInfoDF,on="ruleName",how="left")

reactionWithRuleDF["hasRuleLookup"] = reactionWithRuleDF["ruleSMARTS"].notna()

print(reactionWithRuleDF["hasRuleLookup"].value_counts(dropna=False))
reactionWithRuleDF = reactionWithRuleDF.drop(columns=['sourceDirectory', 'sourceDirectoryPath', 'sourceJsonName', 'sourceJsonPath'])
reactionWithRuleDF

hasRuleLookup
True    176400
Name: count, dtype: int64


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,sourceFolderNum,ruleReactants,ruleProducts,ruleSMARTS,candidateUniProtRaw,numCandidateUniProt,hasRuleLookup
0,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](C=O)O[C@H]1n1cnc...,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,rule0073_5,No_Thermo,"(1, 1, 1)","(1, 1, 1, 1)",Enzymatic,3,4,1,Any;NADH_CoF;O2,NAD_CoF;Any;WATER;WATER,[#6;$([#6&!R]-&!@[#6&R]);!$([#6&!R]-&!@[#6&R]1...,E3VWJ1;F1T282;F1T283;I1TEM3;O94142;Q16850;Q2MJ...,11,True
1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,COC[C@]1(C)O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,Any,Any,([#6;!$([#6&!R]-&!@[#6&!R](-&!@[#6&!R]-&!@[#6&...,A8YZE2,1,True
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,1,Any,Any,([#6;!$([#6&!R](-&!@[#6&!R](-&!@[#6&!R](-&!@[#...,A0A1D3TPC3;A0A1D3TXG7;A0AKV8;A0ALE0;A0B6L2;A0J...,827,True
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,A1JM47;A5I9H6;A6T684;A7MNQ4;A8AJI1;A8GAF7;AKR1...,91,True
4,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,COC(C)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0011_50,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,1,METHYL_DONOR_CoF;Any,METHYL_ACCEPTOR_CoF;Any,[#6:1]-[#16+:2].[#8;$([#8&!R]-[#6&!R]);!$([#8&...,A6XNE5,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176395,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,COC(O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H]...,COC(=O)[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H...,rule0003_177,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,9,NADH_CoF;Any,Any;NAD_CoF,[#6:1]1=[#6:2]-[#7:3]-[#6:4]=[#6:5]-[#6:6]-1.[...,B2ZRE3;H1ZV38;NCED52;O14295;O85057;O94521;P259...,8,True
176396,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,CO[C@@H]1[C@H](OS(=O)(=O)O)[C@@H](C=O)O[C@H]1n...,CO[C@@H]1[C@H](O)[C@@H](C=O)O[C@H]1n1cnc2c(=O)...,rule0048_5,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,9,Any;SULFATE_DONOR_CoF,SULFATE_ACCEPTOR_CoF;Any,[#8;$([#8&!R]-[#6&R]);!$([#8&!R]-[#6&R]1-&@[#6...,A0A0H3L952;A0FIP8;A0QQ53;A1IGX9;A6QNK1;A8XLL3;...,90,True
176397,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,C[C@]12O[C@@]1(CO)O[C@@H](n1cnc3c(=O)[nH]cnc31...,CC(O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@]2(...,rule0028_51,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,9,Any,Any,([#6;!$([#6&!R]-&!@[#6&!R](-&!@[#6&!R]-&!@[#6&...,A8YZE2,1,True
176398,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,CC(=O)O[C@H]1[C@H]2O[C@]2(n2cnc3c(=O)[nH]cnc32...,CC(=O)SCCNC(=O)CCNC(=O)[C@H](O)C(C)(C)COP(=O)(...,rule0062_19,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,9,ACETYL-COA;Any,CoA;Any,[#6:1]-[#16:2].[#8;$([#8&!R]-[#6&R]);!$([#8&!R...,G1EFQ3;O94197;Q5Y9C6;Q94FT4,4,True


### Summarize used rules to understand which rules are most frequent in DORAnet network and how many candidate UniProt proteins each rule has

In [13]:
usedRuleSummaryDF = (
    reactionWithRuleDF[
        reactionWithRuleDF["hasRuleLookup"]
    ]
    .groupby(
        [
            "ruleName",
            "reactionType",
            "ruleReactants",
            "ruleProducts",
            "ruleSMARTS",
            "candidateUniProtRaw",
            "numCandidateUniProt",
        ],
        dropna=False
    )
    .agg(
        numReactions=("reactionString", "count"),
        numStarterSources=("sourceFolderNum", "nunique"),
        exampleReaction=("reactionString", "first"),
        exampleReactants=("reactants", "first"),
        exampleProducts=("products", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)


print(f"Used matched rules: {len(usedRuleSummaryDF):,}")
usedRuleSummaryDF[
    [
        "ruleName",
        "reactionType",
        "numReactions",
        "numCandidateUniProt",
        "exampleReaction",
    ]
].head(20)

Used matched rules: 168


,ruleName,reactionType,numReactions,numCandidateUniProt,exampleReaction
140,rule0169_1,Enzymatic,30900,1602,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...
128,rule0126_2,Enzymatic,21580,72,CC(=O)O[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n...
75,rule0028_51,Enzymatic,17020,1,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...
37,rule0011_51,Enzymatic,6360,1,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c...
74,rule0028_50,Enzymatic,5460,827,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...
99,rule0062_19,Enzymatic,5280,4,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@@]...
85,rule0043_12,Enzymatic,3680,17,CCO[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c(=O)...
16,rule0004_13,Enzymatic,3620,67,CCO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O...
137,rule0165_2,Enzymatic,3580,228,CC(O)(C#N)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH...
36,rule0011_50,Enzymatic,3580,1,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...


## Build one row per rule–UniProt candidate

This converts: `ruleName → "P12345;Q9XYZ1;...`

into: `ruleName | uniprotAccession`

In [14]:
usedRuleSet = set(
    reactionWithRuleDF.loc[
        reactionWithRuleDF["hasRuleLookup"],
        "ruleName"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

usedRuleInfoDF = ruleInfoDF[
    ruleInfoDF["ruleName"].isin(usedRuleSet)
].copy()

ruleCandidateRecords = []

for _, row in usedRuleInfoDF.iterrows():
    candidateList = splitUniProtIds(row["candidateUniProtRaw"])

    for rankIndex, uniprotAccession in enumerate(candidateList, start=1):
        ruleCandidateRecords.append({
            "ruleName": row["ruleName"],
            "candidateRankInRuleFile": rankIndex,
            "uniprotAccession": uniprotAccession,
            "ruleReactants": row["ruleReactants"],
            "ruleProducts": row["ruleProducts"],
            "ruleSMARTS": row["ruleSMARTS"],
        })

usedRuleUniProtCandidateDF = pd.DataFrame(ruleCandidateRecords)

usedRuleUniProtCandidateDF.to_csv(
    os.path.join(DNADesignResultsDir, "usedRuleUniProtCandidateDF.csv"),
    index=False
)

print(f"Used rule-UniProt candidate rows: {len(usedRuleUniProtCandidateDF):,}")
usedRuleUniProtCandidateDF

Used rule-UniProt candidate rows: 26,036


,ruleName,candidateRankInRuleFile,uniprotAccession,ruleReactants,ruleProducts,ruleSMARTS
0,rule0001_86,1,AKR1C1,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
1,rule0001_86,2,C0GBH7,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
2,rule0001_86,3,C0GBH8,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
3,rule0001_86,4,C0RLG0,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
4,rule0001_86,5,C4IVJ1,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
...,...,...,...,...,...,...
26031,rule0846_2,1,Q81G39,Any;PPI;PYROPHOSPHATE_ACCEPTOR_CoF,Any;Any;PYROPHOSPHATE_DONOR_CoF,[#6;$([#6&!R]-&!@[#6&!R]-&!@[#6&!R]);!$([#6&R]...
26032,rule1032_2,1,P31434,Any;WATER,Any;Any,([#8;$([#8&!R]-&!@[#6&!R]):1].[#6:2]-[#8;$([#8...
26033,rule1032_2,2,Q9P999,Any;WATER,Any;Any,([#8;$([#8&!R]-&!@[#6&!R]):1].[#6:2]-[#8;$([#8...
26034,rule1125_1,1,P31434,Any;HF,Any;WATER,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...


### Select subset of UniProt candidates for metadata retrieval

Some DORAnet rules can map to hundreds or thousands of UniProt candidates. For a first query only the top `maxCandidatesPerRule` candidates per rule need to be selcted.

In [15]:
maxCandidatesPerRule = 50

candidateSubsetDF = (
    usedRuleUniProtCandidateDF
    .sort_values(["ruleName", "candidateRankInRuleFile"])
    .groupby("ruleName")
    .head(maxCandidatesPerRule)
    .reset_index(drop=True)
)

print(f"Rules represented            : {candidateSubsetDF['ruleName'].nunique():,}")
print(f"Candidate rows to query      : {len(candidateSubsetDF):,}")
print(f"Unique UniProt IDs to query  : {candidateSubsetDF['uniprotAccession'].nunique():,}")
candidateSubsetDF

Rules represented            : 158
Candidate rows to query      : 4,745
Unique UniProt IDs to query  : 3,651


,ruleName,candidateRankInRuleFile,uniprotAccession,ruleReactants,ruleProducts,ruleSMARTS
0,rule0001_86,1,AKR1C1,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
1,rule0001_86,2,C0GBH7,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
2,rule0001_86,3,C0GBH8,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
3,rule0001_86,4,C0RLG0,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
4,rule0001_86,5,C4IVJ1,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...
...,...,...,...,...,...,...
4740,rule0846_2,1,Q81G39,Any;PPI;PYROPHOSPHATE_ACCEPTOR_CoF,Any;Any;PYROPHOSPHATE_DONOR_CoF,[#6;$([#6&!R]-&!@[#6&!R]-&!@[#6&!R]);!$([#6&R]...
4741,rule1032_2,1,P31434,Any;WATER,Any;Any,([#8;$([#8&!R]-&!@[#6&!R]):1].[#6:2]-[#8;$([#8...
4742,rule1032_2,2,Q9P999,Any;WATER,Any;Any,([#8;$([#8&!R]-&!@[#6&!R]):1].[#6:2]-[#8;$([#8...
4743,rule1125_1,1,P31434,Any;HF,Any;WATER,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...


## 6. Query UniProt for EC numbers, protein names, organisms and sequences

```text
DORAnet rule ID → UniProt accession → EC number/protein sequence
```

In [16]:
# ------------------------------------------------------------
# 1. Helper functions
# ------------------------------------------------------------

def chunkList(inputList, chunkSize):
    for i in range(0, len(inputList), chunkSize):
        yield inputList[i:i + chunkSize]


def cleanUniProtAccession(x):
    """
    Clean UniProt accession IDs coming from DORAnet/JN3604IMT Comments column.
    Examples:
        UniProtKB:P12345 -> P12345
        P12345-2         -> P12345
    """
    if pd.isna(x):
        return None

    x = str(x).strip()

    # Remove common prefixes
    x = x.replace("UniProtKB:", "")
    x = x.replace("UniProt:", "")
    x = x.replace("uniprot:", "")

    # Keep only first token if accidental text is present
    x = x.split()[0].strip()

    # Use canonical accession rather than isoform-specific accession
    # Example: P12345-2 -> P12345
    x = x.split("-")[0].strip()

    if x == "":
        return None

    return x


# UniProt accession pattern: supports common 6-character and 10-character accessions
uniprotAccessionPattern = re.compile(
    r"^([OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9]([A-Z0-9]{3}[0-9]){1,2})$"
)


def isValidUniProtAccession(x):
    if pd.isna(x):
        return False
    return bool(uniprotAccessionPattern.match(str(x).strip()))


def fetchUniProtMetadataFixed(accessionList, chunkSize=20, sleepSeconds=0.5):
    """
    Query UniProtKB metadata for a list of UniProt accessions.

    Important:
    Use accession_id:{acc}, not accession:{acc}.
    """
    metadataDFList = []
    errorRecords = []

    fields = [
        "accession",
        "reviewed",
        "id",
        "protein_name",
        "gene_names",
        "organism_name",
        "organism_id",
        "ec",
        "length",
        "sequence",
    ]

    accessionChunkList = list(chunkList(accessionList, chunkSize))

    for accessionChunk in tqdm(accessionChunkList, desc="Querying UniProt"):
        query = " OR ".join([f"(accession_id:{acc})" for acc in accessionChunk])

        url = "https://rest.uniprot.org/uniprotkb/search"

        params = {
            "query": query,
            "format": "tsv",
            "fields": ",".join(fields),
            "size": chunkSize,
        }

        try:
            response = requests.get(url, params=params, timeout=120)

            if response.status_code != 200:
                errorRecords.append({
                    "chunkFirstAccession": accessionChunk[0],
                    "chunkSize": len(accessionChunk),
                    "statusCode": response.status_code,
                    "message": response.text[:1000],
                    "requestUrl": response.url,
                })
                continue

            chunkDF = pd.read_csv(StringIO(response.text), sep="\t")

            if len(chunkDF) == 0:
                errorRecords.append({
                    "chunkFirstAccession": accessionChunk[0],
                    "chunkSize": len(accessionChunk),
                    "statusCode": response.status_code,
                    "message": "Request succeeded but returned zero rows.",
                    "requestUrl": response.url,
                })
                continue

            metadataDFList.append(chunkDF)

        except Exception as exc:
            errorRecords.append({
                "chunkFirstAccession": accessionChunk[0],
                "chunkSize": len(accessionChunk),
                "statusCode": None,
                "message": str(exc),
                "requestUrl": None,
            })

        time.sleep(sleepSeconds)

    if metadataDFList:
        metadataDF = pd.concat(metadataDFList, ignore_index=True)
        metadataDF = metadataDF.drop_duplicates()
    else:
        metadataDF = pd.DataFrame()

    errorDF = pd.DataFrame(errorRecords)

    return metadataDF, errorDF


# ------------------------------------------------------------
# 2. Clean candidate UniProt IDs from candidateSubsetDF
# ------------------------------------------------------------

candidateSubsetDF = candidateSubsetDF.copy()

candidateSubsetDF["uniprotAccessionClean"] = (
    candidateSubsetDF["uniprotAccession"]
    .apply(cleanUniProtAccession)
)

candidateSubsetDF["isValidUniProtAccession"] = (
    candidateSubsetDF["uniprotAccessionClean"]
    .apply(isValidUniProtAccession)
)

print("Valid/invalid UniProt accession counts:")
print(candidateSubsetDF["isValidUniProtAccession"].value_counts(dropna=False))


# Save invalid UniProt accessions for inspection
invalidUniProtDF = candidateSubsetDF[
    ~candidateSubsetDF["isValidUniProtAccession"]
].copy()

invalidUniProtDF.to_csv(
    os.path.join(sequenceResultsDir, "invalidUniProtAccessionsDF.csv"),
    index=False
)

print(f"Invalid UniProt rows saved: {len(invalidUniProtDF):,}")


# ------------------------------------------------------------
# 3. Build clean unique UniProt accession list
# ------------------------------------------------------------

uniqueAccessionList = sorted(
    candidateSubsetDF.loc[
        candidateSubsetDF["isValidUniProtAccession"],
        "uniprotAccessionClean"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print(f"Valid unique UniProt accessions to query: {len(uniqueAccessionList):,}")


# ------------------------------------------------------------
# 4. Test the UniProt query on one accession first
# ------------------------------------------------------------

if len(uniqueAccessionList) == 0:
    raise RuntimeError("No valid UniProt accessions found after cleaning.")

testAccessionList = uniqueAccessionList[:1]

testMetadataDF, testErrorDF = fetchUniProtMetadataFixed(
    accessionList=testAccessionList,
    chunkSize=1,
    sleepSeconds=0.2,
)

print("\nTest query result:")
print(f"Test metadata rows: {len(testMetadataDF):,}")
print(f"Test error rows   : {len(testErrorDF):,}")

if len(testErrorDF) > 0:
    print("\nTest error message:")
    print(testErrorDF[["statusCode", "message"]].head(1).to_string(index=False))
    raise RuntimeError("UniProt test query failed. Check the error message above.")

display(testMetadataDF)


# ------------------------------------------------------------
# 5. Run the full UniProt metadata query
# ------------------------------------------------------------

uniprotMetadataRawDF, uniprotQueryErrorDF = fetchUniProtMetadataFixed(
    accessionList=uniqueAccessionList,
    chunkSize=20,
    sleepSeconds=0.5,
)

uniprotMetadataRawDF.to_csv(
    os.path.join(sequenceResultsDir, "uniprotMetadataRawDF.csv"),
    index=False
)

if len(uniprotQueryErrorDF) > 0:
    uniprotQueryErrorDF.to_csv(
        os.path.join(sequenceResultsDir, "uniprotQueryErrorDF.csv"),
        index=False
    )

print("\nFull query result:")
print(f"Retrieved UniProt metadata rows: {len(uniprotMetadataRawDF):,}")
print(f"Query error chunks             : {len(uniprotQueryErrorDF):,}")

if len(uniprotQueryErrorDF) > 0:
    print("\nExample query errors:")
    display(uniprotQueryErrorDF.head())

display(uniprotMetadataRawDF.head())


# ------------------------------------------------------------
# 6. Standardize UniProt metadata column names
# ------------------------------------------------------------

uniprotMetadataDF = uniprotMetadataRawDF.copy()

columnRenameDict = {
    "Entry": "uniprotAccession",
    "Reviewed": "uniprotReviewed",
    "Entry Name": "entryName",
    "Protein names": "proteinName",
    "Gene Names": "geneNames",
    "Organism": "organism",
    "Organism (ID)": "organismTaxId",
    "EC number": "ecNumber",
    "Length": "proteinLengthAa",
    "Sequence": "proteinSequence",
}

uniprotMetadataDF = uniprotMetadataDF.rename(columns=columnRenameDict)

requiredUniProtCols = [
    "uniprotAccession",
    "uniprotReviewed",
    "entryName",
    "proteinName",
    "geneNames",
    "organism",
    "organismTaxId",
    "ecNumber",
    "proteinLengthAa",
    "proteinSequence",
]

for colName in requiredUniProtCols:
    if colName not in uniprotMetadataDF.columns:
        uniprotMetadataDF[colName] = np.nan

uniprotMetadataDF = uniprotMetadataDF[requiredUniProtCols].copy()

uniprotMetadataDF.to_csv(
    os.path.join(sequenceResultsDir, "uniprotMetadataDF.csv"),
    index=False
)

print("\nStandardized UniProt metadata:")
print(uniprotMetadataDF.shape)
display(uniprotMetadataDF.head())


# ------------------------------------------------------------
# 7. Merge UniProt metadata back to rule candidates
# ------------------------------------------------------------

candidateSubsetCleanDF = candidateSubsetDF[
    candidateSubsetDF["isValidUniProtAccession"]
].copy()

candidateSubsetCleanDF = candidateSubsetCleanDF.rename(columns={
    "uniprotAccession": "uniprotAccessionOriginal",
    "uniprotAccessionClean": "uniprotAccession",
})

ruleCandidateAnnotatedDF = candidateSubsetCleanDF.merge(
    uniprotMetadataDF,
    on="uniprotAccession",
    how="left"
)

ruleCandidateAnnotatedDF["hasEcNumber"] = (
    ruleCandidateAnnotatedDF["ecNumber"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

ruleCandidateAnnotatedDF["hasProteinSequence"] = (
    ruleCandidateAnnotatedDF["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

ruleCandidateAnnotatedDF.to_csv(
    os.path.join(DNADesignResultsDir, "ruleCandidateAnnotatedDF.csv"),
    index=False
)

print("\nAnnotated rule-candidate summary:")
print("EC number availability:")
print(ruleCandidateAnnotatedDF["hasEcNumber"].value_counts(dropna=False))

print("\nProtein sequence availability:")
print(ruleCandidateAnnotatedDF["hasProteinSequence"].value_counts(dropna=False))

display(ruleCandidateAnnotatedDF.head())


# ------------------------------------------------------------
# 8. Optional: summarize missing UniProt metadata
# ------------------------------------------------------------

missingMetadataDF = ruleCandidateAnnotatedDF[
    ruleCandidateAnnotatedDF["proteinName"].isna()
].copy()

missingMetadataDF.to_csv(
    os.path.join(sequenceResultsDir, "missingUniProtMetadataDF.csv"),
    index=False
)

print(f"\nRows with missing UniProt metadata: {len(missingMetadataDF):,}")

Valid/invalid UniProt accession counts:
isValidUniProtAccession
True     4699
False      46
Name: count, dtype: int64
Invalid UniProt rows saved: 46
Valid unique UniProt accessions to query: 3,632


Querying UniProt: 100%|██████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.07it/s]


Test query result:
Test metadata rows: 1
Test error rows   : 0


,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Organism (ID),EC number,Length,Sequence
0,A0A037YIF5,unreviewed,A0A037YIF5_ECOLX,D-psicose 3-epimerase (EC 5.1.3.30) (D-tagatos...,ycjR ACU57_03015 BGM66_001154 BMT91_09670 BvCm...,Escherichia coli,562,5.1.3.30,262,MKIGTQNQAFFPENILEKFRYIKEMGFDGFEIDGKLLVNNIEEVKA...


Querying UniProt: 100%|██████████████████████████████████████████████████████████████████████████████████| 182/182 [04:10<00:00,  1.38s/it]

...


Standardized UniProt metadata:
(3632, 10)


,uniprotAccession,uniprotReviewed,entryName,proteinName,geneNames,organism,organismTaxId,ecNumber,proteinLengthAa,proteinSequence
0,A0A095AMW7,reviewed,MLES_LEUME,Malolactic enzyme (MLE) (EC 4.1.1.101),mleS LH61_04880,Leuconostoc mesenteroides,1245.0,4.1.1.101,542.0,MNTTGYDILRNPFLNKGTAFSEAERQQLGLTGTLPSQIQTIEEQAE...
1,A0A068Q609,reviewed,C7124_PRUMU,Phenylacetaldehyde oxime monooxygenase CYP71AN...,CYP71AN24,Prunus mume (Japanese apricot) (Armeniaca mume),102107.0,1.14.14.44; 1.14.14.77,526.0,MALLTLFNQIWQEGQLQSSTSSFNIFLVPILCLSIFILFSLTRSSS...
2,A0A060TBM3,unreviewed,A0A060TBM3_BLAAD,alcohol dehydrogenase (EC 1.1.1.1),AADH1 GNLVRS02_ARAD1B16786g,Blastobotrys adeninivorans (Yeast) (Arxula ade...,409370.0,1.1.1.1,348.0,MSIPKTQKAVVFDKNGGPLTYKDIPVPEPADDQILINVKYSGVCHT...
3,A0A068Q816,unreviewed,A0A068Q816_CHICK,Histidine decarboxylase (EC 4.1.1.22),HDC,Gallus gallus (Chicken),9031.0,4.1.1.22,664.0,MEPEEYRRRGKEMVDYICQYLSNVRERRVTPDVQPGYMRAQLPDSA...
4,A0A075Q354,unreviewed,A0A075Q354_GLUDI,pyruvate decarboxylase (EC 4.1.1.1),pdc,Gluconacetobacter diazotrophicus (Acetobacter ...,33996.0,4.1.1.1,558.0,MTYTVGRYLADRLAQIGLKHHFAVAGDYNLVLLDQLLLNTDMQQIY...



Annotated rule-candidate summary:
EC number availability:
hasEcNumber
True     4265
False     434
Name: count, dtype: int64

Protein sequence availability:
hasProteinSequence
True     4594
False     105
Name: count, dtype: int64


,ruleName,candidateRankInRuleFile,uniprotAccessionOriginal,ruleReactants,ruleProducts,ruleSMARTS,uniprotAccession,isValidUniProtAccession,uniprotReviewed,entryName,proteinName,geneNames,organism,organismTaxId,ecNumber,proteinLengthAa,proteinSequence,hasEcNumber,hasProteinSequence
0,rule0001_86,2,C0GBH7,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C0GBH7,True,unreviewed,C0GBH7_9HYPH,"Protocatechuate 3,4-dioxygenase, beta subunit ...",pcaH BCETI_7000097,Brucella ceti str. Cudo,595497.0,2.3.1.72,189.0,MLNELDNDLILNYARPGEMPVGPRILVHGRVLDEGNRPVPGALLEF...,True,True
1,rule0001_86,3,C0GBH8,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C0GBH8,True,unreviewed,C0GBH8_9HYPH,"Protocatechuate 3,4-dioxygenase, alpha subunit...",pcaG BCETI_7000098,Brucella ceti str. Cudo,595497.0,2.3.1.72,214.0,MKTARRATDMVQPLGYLKETASQTAGPYVHIGLTPNFVGINGVFAE...,True,True
2,rule0001_86,4,C0RLG0,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C0RLG0,True,unreviewed,C0RLG0_BRUMB,"Protocatechuate 3,4-dioxygenase, alpha subunit...",pcaG BMEA_B0619,Brucella melitensis biotype 2 (strain ATCC 23457),546272.0,2.3.1.72,205.0,MVQPLGYLKETASQTAGPYVHIGLTPNFVGINGVFAEDLGTGPLYN...,True,True
3,rule0001_86,5,C4IVJ1,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C4IVJ1,True,NaN,C4IVJ1_BRUAO,deleted,NaN,NaN,NaN,NaN,NaN,NaN,False,False
4,rule0001_86,6,C4WLU4,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C4WLU4,True,unreviewed,C4WLU4_9HYPH,"Protocatechuate 3,4-dioxygenase, beta subunit ...",pcaH OINT_2000813,Brucella intermedia LMG 3301,641118.0,2.3.1.72,246.0,MKNSLPETTPFFARDLSMHPPAYTPWYKTSVLRSPTRALLSLEGTK...,True,True



Rows with missing UniProt metadata: 0


In [17]:
ruleCandidateAnnotatedDF

,ruleName,candidateRankInRuleFile,uniprotAccessionOriginal,ruleReactants,ruleProducts,ruleSMARTS,uniprotAccession,isValidUniProtAccession,uniprotReviewed,entryName,proteinName,geneNames,organism,organismTaxId,ecNumber,proteinLengthAa,proteinSequence,hasEcNumber,hasProteinSequence
0,rule0001_86,2,C0GBH7,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C0GBH7,True,unreviewed,C0GBH7_9HYPH,"Protocatechuate 3,4-dioxygenase, beta subunit ...",pcaH BCETI_7000097,Brucella ceti str. Cudo,595497.0,2.3.1.72,189.0,MLNELDNDLILNYARPGEMPVGPRILVHGRVLDEGNRPVPGALLEF...,True,True
1,rule0001_86,3,C0GBH8,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C0GBH8,True,unreviewed,C0GBH8_9HYPH,"Protocatechuate 3,4-dioxygenase, alpha subunit...",pcaG BCETI_7000098,Brucella ceti str. Cudo,595497.0,2.3.1.72,214.0,MKTARRATDMVQPLGYLKETASQTAGPYVHIGLTPNFVGINGVFAE...,True,True
2,rule0001_86,4,C0RLG0,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C0RLG0,True,unreviewed,C0RLG0_BRUMB,"Protocatechuate 3,4-dioxygenase, alpha subunit...",pcaG BMEA_B0619,Brucella melitensis biotype 2 (strain ATCC 23457),546272.0,2.3.1.72,205.0,MVQPLGYLKETASQTAGPYVHIGLTPNFVGINGVFAEDLGTGPLYN...,True,True
3,rule0001_86,5,C4IVJ1,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C4IVJ1,True,NaN,C4IVJ1_BRUAO,deleted,NaN,NaN,NaN,NaN,NaN,NaN,False,False
4,rule0001_86,6,C4WLU4,Any;Any,Any;Any,[#6;$([#6&!R]=&!@[#8&!R]):1]-[#8:2].[#8;$([#8&...,C4WLU4,True,unreviewed,C4WLU4_9HYPH,"Protocatechuate 3,4-dioxygenase, beta subunit ...",pcaH OINT_2000813,Brucella intermedia LMG 3301,641118.0,2.3.1.72,246.0,MKNSLPETTPFFARDLSMHPPAYTPWYKTSVLRSPTRALLSLEGTK...,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4694,rule0846_2,1,Q81G39,Any;PPI;PYROPHOSPHATE_ACCEPTOR_CoF,Any;Any;PYROPHOSPHATE_DONOR_CoF,[#6;$([#6&!R]-&!@[#6&!R]-&!@[#6&!R]);!$([#6&R]...,Q81G39,True,reviewed,DLTA_BACCR,D-alanine--D-alanyl carrier protein ligase (DC...,dltA BC_1372,Bacillus cereus (strain ATCC 14579 / DSM 31 / ...,226900.0,6.2.1.54,504.0,MKLLEQIEKWAAETPDQTAFVWRDAKITYKQLKEDSDALAHWISSE...,True,True
4695,rule1032_2,1,P31434,Any;WATER,Any;Any,([#8;$([#8&!R]-&!@[#6&!R]):1].[#6:2]-[#8;$([#8...,P31434,True,reviewed,XYLS_ECOLI,Alpha-xylosidase (EC 3.2.1.177),yicI b3656 JW3631,Escherichia coli (strain K12),83333.0,3.2.1.177,772.0,MKISDGNWLIQPGLNLIHPLQVFEVEQQDNEMVVYAAPRDVRERTW...,True,True
4696,rule1032_2,2,Q9P999,Any;WATER,Any;Any,([#8;$([#8&!R]-&!@[#6&!R]):1].[#6:2]-[#8;$([#8...,Q9P999,True,reviewed,XYLS_SACS2,Alpha-xylosidase (EC 3.2.1.177),xylS SSO3022,Saccharolobus solfataricus (strain ATCC 35092 ...,273057.0,3.2.1.177,731.0,MRIGNLNVEIEFIADNIVRVLYYYGREAIVDNSLVVLPNLEKLSIK...,True,True
4697,rule1125_1,1,P31434,Any;HF,Any;WATER,[#6;$([#6&R]1-&@[#8&R]-&@[#6&R]-&@[#6&R](-&@[#...,P31434,True,reviewed,XYLS_ECOLI,Alpha-xylosidase (EC 3.2.1.177),yicI b3656 JW3631,Escherichia coli (strain K12),83333.0,3.2.1.177,772.0,MKISDGNWLIQPGLNLIHPLQVFEVEQQDNEMVVYAAPRDVRERTW...,True,True


## 7. Summarize EC numbers for each DORAnet rule to understand what `enzyme classes` are associated with each DORAnet rule

In [18]:
def uniqueSemicolonValues(series):
    valueSet = set()

    for value in series.dropna():
        for part in str(value).split(";"):
            part = part.strip()
            if part:
                valueSet.add(part)

    return ";".join(sorted(valueSet))


def firstNonEmptyValues(series, maxValues=5):
    valueList = []

    for value in series.dropna():
        value = str(value).strip()
        if value and value not in valueList:
            valueList.append(value)
        if len(valueList) >= maxValues:
            break

    return " | ".join(valueList)


ruleEcSummaryDF = (
    ruleCandidateAnnotatedDF
    .groupby("ruleName")
    .agg(
        numCandidateQueried=("uniprotAccession", "nunique"),
        numWithEc=("hasEcNumber", "sum"),
        numWithProteinSequence=("hasProteinSequence", "sum"),
        ecNumberList=("ecNumber", uniqueSemicolonValues),
        exampleProteinNames=("proteinName", firstNonEmptyValues),
        exampleOrganisms=("organism", firstNonEmptyValues),
        ruleReactants=("ruleReactants", "first"),
        ruleProducts=("ruleProducts", "first"),
        ruleSMARTS=("ruleSMARTS", "first"),
    )
    .reset_index()
)

# Add reaction-frequency information from your DORAnet network
usedRuleFrequencyDF = (
    reactionWithRuleDF
    .groupby("ruleName")
    .agg(
        numDoranetReactions=("reactionString", "count"),
        numStarterSources=("sourceFolderNum", "nunique"),
        exampleDoranetReaction=("reactionString", "first"),
        reactionType=("reactionType", "first"),
    )
    .reset_index()
)

ruleEcSummaryDF = ruleEcSummaryDF.merge(
    usedRuleFrequencyDF,
    on="ruleName",
    how="left"
)

ruleEcSummaryDF = ruleEcSummaryDF.sort_values(
    "numDoranetReactions",
    ascending=False
)

ruleEcSummaryDF.to_csv(
    os.path.join(DNADesignResultsDir, "ruleEcSummaryDF.csv"),
    index=False
)

print(f"Rules summarized: {len(ruleEcSummaryDF):,}")

ruleEcSummaryDF[
    [
        "ruleName",
        "reactionType",
        "numDoranetReactions",
        "numCandidateQueried",
        "numWithEc",
        "ecNumberList",
        "exampleProteinNames",
        "exampleDoranetReaction",
    ]
].head(30)

Rules summarized: 158


,ruleName,reactionType,numDoranetReactions,numCandidateQueried,numWithEc,ecNumberList,exampleProteinNames,exampleDoranetReaction
134,rule0169_1,Enzymatic,30900,50,50,5.3.1.-;5.3.1.14;5.3.1.23;5.3.1.24;5.3.1.25;5....,N-(5'-phosphoribosyl)anthranilate isomerase (E...,CC(=O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc...
122,rule0126_2,Enzymatic,21580,50,50,1.1.99.24;2.1.3.1;2.2.1.3;2.7.8.29;2.8.3.16;2....,Formyl-CoA:oxalate CoA-transferase (FCOCT) (EC...,CC(=O)O[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n...
72,rule0028_51,Enzymatic,17020,1,1,5.1.3.9,Putative N-acetylmannosamine-6-phosphate 2-epi...,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...
35,rule0011_51,Enzymatic,6360,1,1,2.1.1.315,27-O-demethylrifamycin SV methyltransferase (D...,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c...
71,rule0028_50,Enzymatic,5460,50,49,5.4.2.11;5.4.2.12,"2,3-bisphosphoglycerate-independent phosphogly...",O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...
94,rule0062_19,Enzymatic,5280,4,2,2.3.1.-;2.3.1.150,Alcohol acyl transferase | Trichothecene 3-O-a...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@@]...
81,rule0043_12,Enzymatic,3680,17,10,2.1.1.243;2.1.1.255;2.1.1.281;2.1.1.317,Geranyl diphosphate 2-C-methyltransferase (GPP...,CCO[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c(=O)...
16,rule0004_13,Enzymatic,3620,50,40,1.-.-.-;1.14.13.181;1.14.13.227;1.14.13.229;1....,Cytochrome P450 4V2 (Docosahexaenoic acid omeg...,CCO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O...
34,rule0011_50,Enzymatic,3580,1,0,,Coniferyl alcohol 9-O-methyltransferase,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...
131,rule0165_2,Enzymatic,3580,50,49,1.1.-.-;1.1.1.192;1.1.1.23;1.1.1.6;3.5.4.19;3....,Histidine biosynthesis trifunctional protein [...,CC(O)(C#N)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH...


### Rank candidate enzymes per rule

Each rule may have many UniProt candidates. Rank them in certain order

In [19]:
candidateRankDF = ruleCandidateAnnotatedDF.copy()

candidateRankDF["isReviewed"] = (
    candidateRankDF["uniprotReviewed"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains("reviewed|swiss-prot", na=False)
)

candidateRankDF["hasEcNumber"] = (
    candidateRankDF["ecNumber"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

candidateRankDF["hasProteinSequence"] = (
    candidateRankDF["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

candidateRankDF["proteinLengthAaNumeric"] = pd.to_numeric(
    candidateRankDF["proteinLengthAa"],
    errors="coerce"
)

candidateRankDF["hasReasonableLength"] = (
    candidateRankDF["proteinLengthAaNumeric"]
    .fillna(0)
    .between(100, 1500)
)

candidateRankDF["isBacterialProtein"] = (
    candidateRankDF["organism"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.contains("bacter|escherichia|bacillus|pseudomonas|streptomyces|salmonella|klebsiella", na=False)
)

candidateRankDF["candidateScore"] = (
    candidateRankDF["isReviewed"].astype(int) * 4
    + candidateRankDF["hasEcNumber"].astype(int) * 3
    + candidateRankDF["hasProteinSequence"].astype(int) * 2
    + candidateRankDF["hasReasonableLength"].astype(int) * 1
    + candidateRankDF["isBacterialProtein"].astype(int) * 1
    - candidateRankDF["candidateRankInRuleFile"] / 10000.0
)

candidateRankDF = candidateRankDF.sort_values(
    ["ruleName", "candidateScore", "candidateRankInRuleFile"],
    ascending=[True, False, True]
)

topCandidatePerRuleDF = (
    candidateRankDF
    .groupby("ruleName")
    .head(10)
    .reset_index(drop=True)
)

topCandidatePerRuleDF.to_csv(
    os.path.join(DNADesignResultsDir, "topCandidatePerRuleDF.csv"),
    index=False
)

topCandidatePerRuleDF[
    [
        "ruleName",
        "uniprotAccession",
        "candidateScore",
        "uniprotReviewed",
        "ecNumber",
        "proteinName",
        "organism",
        "proteinLengthAa",
        "candidateRankInRuleFile",
    ]
]

,ruleName,uniprotAccession,candidateScore,uniprotReviewed,ecNumber,proteinName,organism,proteinLengthAa,candidateRankInRuleFile
0,rule0001_86,Q9I537,10.9978,unreviewed,2.3.2.3,Phosphatidylglycerol lysyltransferase (EC 2.3....,Pseudomonas aeruginosa (strain ATCC 15692 / DS...,881.0,22
1,rule0001_86,C0GBH7,9.9998,unreviewed,2.3.1.72,"Protocatechuate 3,4-dioxygenase, beta subunit ...",Brucella ceti str. Cudo,189.0,2
2,rule0001_86,C0GBH8,9.9997,unreviewed,2.3.1.72,"Protocatechuate 3,4-dioxygenase, alpha subunit...",Brucella ceti str. Cudo,214.0,3
3,rule0001_86,C0RLG0,9.9996,unreviewed,2.3.1.72,"Protocatechuate 3,4-dioxygenase, alpha subunit...",Brucella melitensis biotype 2 (strain ATCC 23457),205.0,4
4,rule0001_86,C4WLU4,9.9994,unreviewed,2.3.1.72,"Protocatechuate 3,4-dioxygenase, beta subunit ...",Brucella intermedia LMG 3301,246.0,6
...,...,...,...,...,...,...,...,...,...
1301,rule0846_2,Q81G39,10.9999,reviewed,6.2.1.54,D-alanine--D-alanyl carrier protein ligase (DC...,Bacillus cereus (strain ATCC 14579 / DSM 31 / ...,504.0,1
1302,rule1032_2,P31434,10.9999,reviewed,3.2.1.177,Alpha-xylosidase (EC 3.2.1.177),Escherichia coli (strain K12),772.0,1
1303,rule1032_2,Q9P999,9.9998,reviewed,3.2.1.177,Alpha-xylosidase (EC 3.2.1.177),Saccharolobus solfataricus (strain ATCC 35092 ...,731.0,2
1304,rule1125_1,P31434,10.9999,reviewed,3.2.1.177,Alpha-xylosidase (EC 3.2.1.177),Escherichia coli (strain K12),772.0,1


### Print Organisms: `E.coli/Pseudomonas putida` etc

In [20]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"].astype(str).str.contains("Escherichia coli", na=False),
        "organism"].dropna().astype(str).unique())

for org in queryOrganisms:
    print(org)

Escherichia coli
Escherichia coli (strain 55989 / EAEC)
Escherichia coli (strain ATCC 8739 / DSM 1576 / NBRC 3972 / NCIMB 8545 / WDCM 00012 / Crooks)
Escherichia coli (strain K12 / DH10B)
Escherichia coli (strain K12)
Escherichia coli (strain SE11)
Escherichia coli (strain SMS-3-5 / SECEC)
Escherichia coli (strain UTI89 / UPEC)
Escherichia coli O139:H28 (strain E24377A / ETEC)
Escherichia coli O157:H7
Escherichia coli O157:H7 (strain EC4115 / EHEC)
Escherichia coli O1:K1 / APEC
Escherichia coli O6:H1 (strain CFT073 / ATCC 700928 / UPEC)
Escherichia coli O6:K15:H31 (strain 536 / UPEC)
Escherichia coli O9:H4 (strain HS)


In [21]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"].astype(str).str.contains("Pseudomonas putida", na=False),
        "organism"].dropna().astype(str).unique())

for org in queryOrganisms:
    print(org)

Pseudomonas putida (Arthrobacter siderocapsulatus)
Pseudomonas putida (strain ATCC 700007 / DSM 6899 / JCM 31910 / BCRC 17059 / LMG 24140 / F1)
Pseudomonas putida (strain GB-1)


In [22]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"].astype(str).str.contains("Bacillus", na=False),
        "organism"].dropna().astype(str).unique())

for org in queryOrganisms:
    print(org)

Alkalihalophilus pseudofirmus (Bacillus pseudofirmus)
Alkalihalophilus pseudofirmus (strain ATCC BAA-2126 / JCM 17055 / OF4) (Bacillus pseudofirmus)
Bacillus cereus (strain ATCC 14579 / DSM 31 / CCUG 7414 / JCM 2152 / NBRC 15305 / NCIMB 9373 / NCTC 2599 / NRRL B-3711)
Bacillus cytotoxicus (strain DSM 22905 / CIP 110041 / 391-98 / NVH 391-98)
Bacillus licheniformis
Bacillus licheniformis (strain ATCC 14580 / DSM 13 / JCM 2505 / CCUG 7422 / NBRC 12200 / NCIMB 9375 / NCTC 10341 / NRRL NRS-1264 / Gibson 46)
Bacillus mycoides (strain KBAB4) (Bacillus weihenstephanensis)
Bacillus pumilus (strain SAFR-032)
Bacillus sp. (strain OxB-1)
Bacillus sp. (strain YM-1)
Bacillus subtilis
Bacillus subtilis (strain 168)
Bacillus thuringiensis (strain Al Hakam)
Geobacillus stearothermophilus (Bacillus stearothermophilus)
Lysinibacillus sphaericus (Bacillus sphaericus)
Priestia megaterium (strain DSM 319 / IMG 1521) (Bacillus megaterium)


### Check which E. coli organisms are present among top ranked candidates

In [23]:
queryOrganisms = sorted(
    topCandidatePerRuleDF.loc[
        topCandidatePerRuleDF["organism"]
        .astype(str)
        .str.contains("Escherichia coli", na=False),
        "organism"
    ]
    .dropna()
    .astype(str)
    .unique()
)

print(f"Number of distinct E. coli organism labels: {len(queryOrganisms)}")

for org in queryOrganisms:
    print(org)

Number of distinct E. coli organism labels: 15
Escherichia coli
Escherichia coli (strain 55989 / EAEC)
Escherichia coli (strain ATCC 8739 / DSM 1576 / NBRC 3972 / NCIMB 8545 / WDCM 00012 / Crooks)
Escherichia coli (strain K12 / DH10B)
Escherichia coli (strain K12)
Escherichia coli (strain SE11)
Escherichia coli (strain SMS-3-5 / SECEC)
Escherichia coli (strain UTI89 / UPEC)
Escherichia coli O139:H28 (strain E24377A / ETEC)
Escherichia coli O157:H7
Escherichia coli O157:H7 (strain EC4115 / EHEC)
Escherichia coli O1:K1 / APEC
Escherichia coli O6:H1 (strain CFT073 / ATCC 700928 / UPEC)
Escherichia coli O6:K15:H31 (strain 536 / UPEC)
Escherichia coli O9:H4 (strain HS)


### count how many rules have at least one E. coli candidate

In [24]:
topCandidatePerRuleDF["isEscherichiaColi"] = (
    topCandidatePerRuleDF["organism"]
    .fillna("")
    .astype(str)
    .str.contains("Escherichia coli", na=False)
)

ecoliCandidateSummaryDF = (
    topCandidatePerRuleDF
    .groupby("ruleName")
    .agg(
        numTopCandidates=("uniprotAccession", "nunique"),
        numEcoliCandidates=("isEscherichiaColi", "sum"),
    )
    .reset_index()
)

ecoliCandidateSummaryDF["hasEcoliCandidate"] = (
    ecoliCandidateSummaryDF["numEcoliCandidates"] > 0
)

print(ecoliCandidateSummaryDF["hasEcoliCandidate"].value_counts(dropna=False))

ecoliCandidateSummaryDF.head()

hasEcoliCandidate
False    94
True     64
Name: count, dtype: int64


,ruleName,numTopCandidates,numEcoliCandidates,hasEcoliCandidate
0,rule0001_86,10,0,False
1,rule0001_87,10,0,False
2,rule0001_88,3,0,False
3,rule0001_90,9,0,False
4,rule0002_142,10,0,False


### 8. Automatically select the best E. coli enzyme per rule

This selects the highest-scoring E. coli candidate for each DORAnet rule

In [25]:
candidateRankDF["isEscherichiaColi"] = (
    candidateRankDF["organism"]
    .fillna("")
    .astype(str)
    .str.contains("Escherichia coli", na=False)
)

ecoliCandidateDF = candidateRankDF[
    candidateRankDF["isEscherichiaColi"]
].copy()

ecoliCandidateDF = ecoliCandidateDF.sort_values(
    ["ruleName", "candidateScore", "candidateRankInRuleFile"],
    ascending=[True, False, True]
)

selectedEcoliEnzymeDF = (
    ecoliCandidateDF
    .groupby("ruleName")
    .head(1)
    .reset_index(drop=True)
)

selectedEcoliEnzymeDF["selectForDnaDesign"] = "yes"
selectedEcoliEnzymeDF["selectionMode"] = "automatic_ecoli_only"
selectedEcoliEnzymeDF["enzymeConfidence"] = "automatic_needs_later_review"
selectedEcoliEnzymeDF["selectionReason"] = (
    "Automatically selected highest-scoring Escherichia coli candidate per DORAnet rule"
)

selectedEcoliEnzymeDF.to_csv(
    os.path.join(DNADesignResultsDir, "selectedEcoliEnzymeDF.csv"),
    index=False
)

print(f"Rules with selected E. coli enzyme: {selectedEcoliEnzymeDF['ruleName'].nunique():,}")
print(f"Selected enzyme rows             : {len(selectedEcoliEnzymeDF):,}")

selectedEcoliEnzymeDF[
    [
        "ruleName",
        "uniprotAccession",
        "candidateScore",
        "uniprotReviewed",
        "ecNumber",
        "proteinName",
        "organism",
        "proteinLengthAa",
        "selectionMode",
    ]
].head(30)

Rules with selected E. coli enzyme: 68
Selected enzyme rows             : 68


,ruleName,uniprotAccession,candidateScore,uniprotReviewed,ecNumber,proteinName,organism,proteinLengthAa,selectionMode
0,rule0001_87,E0J1Q4,10.9968,reviewed,2.3.1.251,Lipid A palmitoyltransferase PagP (EC 2.3.1.25...,Escherichia coli (strain ATCC 9637 / CCM 2024 ...,186.0,automatic_ecoli_only
1,rule0002_148,P0A9S1,10.9995,reviewed,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),382.0,automatic_ecoli_only
2,rule0003_152,A1A9R3,10.9999,reviewed,1.1.1.298,Probable malonic semialdehyde reductase RutE (...,Escherichia coli O1:K1 / APEC,196.0,automatic_ecoli_only
3,rule0003_171,P0A9S1,10.9995,reviewed,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),382.0,automatic_ecoli_only
4,rule0003_173,P06988,10.9950,reviewed,1.1.1.23,Histidinol dehydrogenase (HDH) (EC 1.1.1.23),Escherichia coli (strain K12),434.0,automatic_ecoli_only
5,rule0003_177,P25906,10.9993,reviewed,1.1.1.65,Pyridoxine 4-dehydrogenase (EC 1.1.1.65),Escherichia coli (strain K12),286.0,automatic_ecoli_only
6,rule0007_174,A1A7K6,10.9999,reviewed,3.1.5.1,Deoxyguanosinetriphosphate triphosphohydrolase...,Escherichia coli O1:K1 / APEC,505.0,automatic_ecoli_only
7,rule0007_193,P13001,10.9975,reviewed,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,automatic_ecoli_only
8,rule0007_198,P13001,10.9950,reviewed,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,automatic_ecoli_only
9,rule0007_200,P13001,10.9996,reviewed,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),256.0,automatic_ecoli_only


### Check which rules do not have E. coli candidates

strict E. coli-only selection may leave some DORAnet rules without enzymes

In [26]:
allUsedRuleSet = set(
    reactionWithRuleDF.loc[
        reactionWithRuleDF["hasRuleLookup"],
        "ruleName"
    ]
    .dropna()
    .astype(str)
    .str.strip()
)

selectedEcoliRuleSet = set(
    selectedEcoliEnzymeDF["ruleName"]
    .dropna()
    .astype(str)
    .str.strip()
)

rulesWithoutEcoliSelection = sorted(
    list(allUsedRuleSet.difference(selectedEcoliRuleSet))
)

rulesWithoutEcoliSelectionDF = (
    reactionWithRuleDF[
        reactionWithRuleDF["ruleName"].isin(rulesWithoutEcoliSelection)
    ]
    .groupby("ruleName")
    .agg(
        numReactions=("reactionString", "count"),
        exampleReaction=("reactionString", "first"),
        reactionType=("reactionType", "first"),
        numCandidateUniProt=("numCandidateUniProt", "first"),
    )
    .reset_index()
    .sort_values("numReactions", ascending=False)
)

rulesWithoutEcoliSelectionDF.to_csv(
    os.path.join(DNADesignResultsDir, "rulesWithoutEcoliSelectionDF.csv"),
    index=False
)

print(f"Rules without E. coli selection: {len(rulesWithoutEcoliSelectionDF):,}")

rulesWithoutEcoliSelectionDF.head(30)

Rules without E. coli selection: 100


,ruleName,numReactions,exampleReaction,reactionType,numCandidateUniProt
43,rule0028_51,17020,COC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@H](O)...,Enzymatic,1
24,rule0011_51,6360,CC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c...,Enzymatic,1
59,rule0062_19,5280,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@@]...,Enzymatic,4
48,rule0043_12,3680,CCO[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2c(=O)...,Enzymatic,17
10,rule0004_13,3620,CCO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O...,Enzymatic,67
23,rule0011_50,3580,CC(O)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,Enzymatic,1
41,rule0028_35,2960,CCC(=O)O[C@@H]1[C@H](O)[C@@H](CO)O[C@H]1n1cnc2...,Enzymatic,10
98,rule0491_2,2920,CO[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32)O[...,Enzymatic,0
13,rule0005_62,2560,Cc1nc2c(=O)[nH]cnc2n1[C@@H]1O[C@H](CO)[C@@H](O...,Enzymatic,13
63,rule0073_5,2520,CO[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n1cnc2...,Enzymatic,11


### Merge selected E. coli enzymes back to DORAnet reactions

In [27]:
selectedEnzymeDF = selectedEcoliEnzymeDF.copy()

selectedEnzymeForMergeDF = selectedEnzymeDF[
    [
        "ruleName",
        "uniprotAccession",
        "ecNumber",
        "proteinName",
        "organism",
        "uniprotReviewed",
        "proteinLengthAa",
        "proteinSequence",
        "selectionReason",
        "enzymeConfidence",
        "selectionMode",
    ]
].copy()

reactionWithSelectedEnzymeDF = reactionWithRuleDF.merge(
    selectedEnzymeForMergeDF,
    on="ruleName",
    how="left"
)

reactionWithSelectedEnzymeDF["hasSelectedEnzyme"] = (
    reactionWithSelectedEnzymeDF["uniprotAccession"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

reactionWithSelectedEnzymeDF.to_csv(
    os.path.join(DNADesignResultsDir, "reactionWithSelectedEcoliEnzymeDF.csv"),
    index=False
)

print(reactionWithSelectedEnzymeDF["hasSelectedEnzyme"].value_counts(dropna=False))

reactionWithSelectedEnzymeDF = reactionWithSelectedEnzymeDF[
    reactionWithSelectedEnzymeDF["uniprotAccession"].notna()
].copy()

reactionWithSelectedEnzymeDF[
    [
        "reactionString",
        "ruleName",
        "ecNumber",
        "uniprotAccession",
        "proteinName",
        "organism",
        "enzymeConfidence",
        "selectionMode",
    ]
].head()

hasSelectedEnzyme
False    94360
True     82040
Name: count, dtype: int64


,reactionString,ruleName,ecNumber,uniprotAccession,proteinName,organism,enzymeConfidence,selectionMode
2,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,5.4.2.11,A1A8Z8,"2,3-bisphosphoglycerate-dependent phosphoglyce...",Escherichia coli O1:K1 / APEC,automatic_needs_later_review,automatic_ecoli_only
3,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,2.3.1.251,E0J1Q4,Lipid A palmitoyltransferase PagP (EC 2.3.1.25...,Escherichia coli (strain ATCC 9637 / CCM 2024 ...,automatic_needs_later_review,automatic_ecoli_only
5,CC(O)(C#N)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH...,rule0165_2,1.1.1.23,P06988,Histidinol dehydrogenase (HDH) (EC 1.1.1.23),Escherichia coli (strain K12),automatic_needs_later_review,automatic_ecoli_only
8,CC(=O)O[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n...,rule0126_2,2.8.3.16,A1ADQ1,Formyl-CoA:oxalate CoA-transferase (FCOCT) (EC...,Escherichia coli O1:K1 / APEC,automatic_needs_later_review,automatic_ecoli_only
9,CC(=O)[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0091_14,1.4.3.21,P46883,Primary amine oxidase (EC 1.4.3.21) (2-phenyle...,Escherichia coli (strain K12),automatic_needs_later_review,automatic_ecoli_only


## Create selected E. coli protein-sequence table

In [28]:
selectedProteinSequenceDF = (
    selectedEcoliEnzymeDF[
        [
            "ruleName",
            "uniprotAccession",
            "ecNumber",
            "proteinName",
            "organism",
            "proteinLengthAa",
            "proteinSequence",
            "selectionMode",
            "selectionReason",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

selectedProteinSequenceDF["hasProteinSequence"] = (
    selectedProteinSequenceDF["proteinSequence"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

selectedProteinSequenceDF.to_csv(
    os.path.join(sequenceResultsDir, "selectedEcoliProteinSequenceDF.csv"),
    index=False
)

print(selectedProteinSequenceDF["hasProteinSequence"].value_counts(dropna=False))

selectedProteinSequenceDF.head()

hasProteinSequence
True    68
Name: count, dtype: int64


,ruleName,uniprotAccession,ecNumber,proteinName,organism,proteinLengthAa,proteinSequence,selectionMode,selectionReason,hasProteinSequence
0,rule0001_87,E0J1Q4,2.3.1.251,Lipid A palmitoyltransferase PagP (EC 2.3.1.25...,Escherichia coli (strain ATCC 9637 / CCM 2024 ...,186.0,MNVSKYVAIFSFVFIQLISVGKVFANADEWMTTFRENIAQTRQQPE...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
1,rule0002_148,P0A9S1,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),382.0,MANRMILNETAWFGRGAVGALTDEVKRRGYQKALIVTDKTLVQCGV...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
2,rule0003_152,A1A9R3,1.1.1.298,Probable malonic semialdehyde reductase RutE (...,Escherichia coli O1:K1 / APEC,196.0,MNEAVSPGALSTLFTDARTHNGWRETPVSDETLREIYALMKWGPTS...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
3,rule0003_171,P0A9S1,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),382.0,MANRMILNETAWFGRGAVGALTDEVKRRGYQKALIVTDKTLVQCGV...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True
4,rule0003_173,P06988,1.1.1.23,Histidinol dehydrogenase (HDH) (EC 1.1.1.23),Escherichia coli (strain K12),434.0,MSFNTIIDWNSCTAEQQRQLLMRPAISASESITRTVNDILDNVKAR...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,True


### Summarize selected E. coli enzymes by EC class

In [29]:
selectedEcoliEcSummaryDF = (
    selectedProteinSequenceDF
    .groupby(["ecNumber", "proteinName", "organism"], dropna=False)
    .agg(
        numRules=("ruleName", "nunique"),
        exampleRule=("ruleName", "first"),
        meanProteinLength=("proteinLengthAa", "mean"),
    )
    .reset_index()
    .sort_values("numRules", ascending=False)
)

selectedEcoliEcSummaryDF.to_csv(
    os.path.join(DNADesignResultsDir, "selectedEcoliEcSummaryDF.csv"),
    index=False
)

selectedEcoliEcSummaryDF

,ecNumber,proteinName,organism,numRules,exampleRule,meanProteinLength
0,1.1.1.23,Histidinol dehydrogenase (HDH) (EC 1.1.1.23),Escherichia coli (strain K12),4,rule0003_173,434.0
27,3.1.1.85,Pimeloyl-[acyl-carrier protein] methyl ester e...,Escherichia coli (strain K12),4,rule0007_193,256.0
4,1.1.5.13,L-2-hydroxyglutarate dehydrogenase (L2HG dehyd...,Escherichia coli (strain K12),2,rule0074_17,422.0
19,2.7.1.15,Ribokinase (RK) (EC 2.7.1.15),Escherichia coli (strain K12),2,rule0014_12,309.0
10,2.1.1.33,tRNA (guanine-N(7)-)-methyltransferase (EC 2.1...,Escherichia coli O1:K1 / APEC,2,rule0120_1,239.0
3,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),2,rule0002_148,382.0
30,3.1.4.55,"Phosphoribosyl 1,2-cyclic phosphate phosphodie...",Escherichia coli (strain K12),2,rule0032_09,252.0
33,3.2.1.177,Alpha-xylosidase (EC 3.2.1.177),Escherichia coli (strain K12),2,rule1032_2,772.0
22,2.7.1.71,Shikimate kinase 2 (SK 2) (EC 2.7.1.71),Escherichia coli O1:K1 / APEC,2,rule0014_29,174.0
8,1.4.3.21,Primary amine oxidase (EC 1.4.3.21) (2-phenyle...,Escherichia coli (strain K12),2,rule0091_14,757.0


### Codon-optimize selected enzymes for E. coli

In [30]:
targetSpecies = "e_coli"

In [31]:
from Bio.Data import CodonTable

from dnachisel import (
    DnaOptimizationProblem,
    EnforceTranslation,
    EnforceGCContent,
    AvoidPattern,
    MaximizeCAI,
)



minGc = 0.30
maxGc = 0.70
gcWindow = 50
stopCodon = "TAA"

forbiddenPatternList = [
    "BsaI_site",
    "BsmBI_site",
    "EcoRI_site",
    "XbaI_site",
    "SpeI_site",
    "PstI_site",
    "NotI_site",
]


def buildSimpleReverseCodonMap():
    standardTable = CodonTable.unambiguous_dna_by_name["Standard"]

    aaToCodons = {}
    for codon, aa in standardTable.forward_table.items():
        aaToCodons.setdefault(aa, []).append(codon)

    preferredCodonDict = {
        aa: sorted(codons)[0]
        for aa, codons in aaToCodons.items()
    }

    preferredCodonDict["M"] = "ATG"
    preferredCodonDict["W"] = "TGG"

    return preferredCodonDict


def reverseTranslateProtein(proteinSequence):
    preferredCodonDict = buildSimpleReverseCodonMap()

    cleanProteinSequence = (
        str(proteinSequence)
        .replace("*", "")
        .replace(" ", "")
        .replace("\n", "")
        .upper()
    )

    codonList = []

    for aa in cleanProteinSequence:
        if aa not in preferredCodonDict:
            raise ValueError(f"Cannot reverse translate amino acid: {aa}")
        codonList.append(preferredCodonDict[aa])

    return "".join(codonList)


def optimizeCdsForEcoli(initialCds):
    sequenceLength = len(initialCds)
    geneLocation = (0, sequenceLength)

    constraints = [
        EnforceTranslation(location=geneLocation),
        EnforceGCContent(mini=minGc, maxi=maxGc, window=gcWindow),
    ]

    for patternName in forbiddenPatternList:
        constraints.append(AvoidPattern(patternName))

    objectives = [
        MaximizeCAI(species=targetSpecies, location=geneLocation)
    ]

    optimizationProblem = DnaOptimizationProblem(
        sequence=initialCds,
        constraints=constraints,
        objectives=objectives,
    )

    optimizationProblem.resolve_constraints()
    optimizationProblem.optimize()

    return optimizationProblem.sequence, optimizationProblem


proteinForDesignDF = selectedProteinSequenceDF[
    selectedProteinSequenceDF["hasProteinSequence"]
].copy()

optimizedGeneRecords = []

for _, row in tqdm(
    proteinForDesignDF.iterrows(),
    total=len(proteinForDesignDF),
    desc="Optimizing E. coli selected genes"
):
    ruleName = row["ruleName"]
    uniprotAccession = row["uniprotAccession"]
    proteinSequence = row["proteinSequence"]

    initialCds = reverseTranslateProtein(proteinSequence)
    optimizedCds, optimizationProblem = optimizeCdsForEcoli(initialCds)
    finalCds = optimizedCds + stopCodon

    optimizedGeneRecords.append({
        "ruleName": ruleName,
        "uniprotAccession": uniprotAccession,
        "ecNumber": row["ecNumber"],
        "proteinName": row["proteinName"],
        "sourceOrganism": row["organism"],
        "expressionHost": "Escherichia coli",
        "targetSpecies": targetSpecies,
        "proteinLengthAa": len(str(proteinSequence)),
        "initialCdsLengthBp": len(initialCds),
        "optimizedCdsLengthBp": len(finalCds),
        "optimizedCds": finalCds,
        "selectionMode": row["selectionMode"],
        "selectionReason": row["selectionReason"],
        "constraintsSummary": optimizationProblem.constraints_text_summary(),
        "objectivesSummary": optimizationProblem.objectives_text_summary(),
    })

optimizedEcoliGeneDF = pd.DataFrame(optimizedGeneRecords)

optimizedEcoliGeneDF.to_csv(
    os.path.join(dnaResultsDir, "optimizedEcoliGeneDF.csv"),
    index=False
)

with open(os.path.join(dnaResultsDir, "optimizedEcoliGenes.fasta"), "w", encoding="utf-8") as f:
    for _, row in optimizedEcoliGeneDF.iterrows():
        fastaHeader = (
            f">{row['ruleName']}|{row['uniprotAccession']}|"
            f"EC={row['ecNumber']}|host=Escherichia_coli"
        )
        f.write(fastaHeader + "\n")
        f.write(str(row["optimizedCds"]) + "\n\n")

print(f"Optimized genes: {len(optimizedEcoliGeneDF):,}")

optimizedEcoliGeneDF.head()

constraint:   0%|                                                                  | 0/8 [00:00<?, ?it/s, now=AvoidPattern[0-558](patte...]

location:   0%|                                                                                            | 0/1 [00:00<?, ?it/s, now=None]

location:   0%|                                                                                      | 0/1 [00:00<?, ?it/s, now=353-359(-)]

                                                                                                                                           
                                                                                                                                           
objective:   0%|                                                                   | 0/1 [00:00<?, ?it/s, now=MaximizeCAI[0-558](e_coli...]

location:   0%|                                                                                    | 0/135 [00:00<?, ?it/s, now=353-359(-)]

location:   0%|

Optimized genes: 68


,ruleName,uniprotAccession,ecNumber,proteinName,sourceOrganism,expressionHost,targetSpecies,proteinLengthAa,initialCdsLengthBp,optimizedCdsLengthBp,optimizedCds,selectionMode,selectionReason,constraintsSummary,objectivesSummary
0,rule0001_87,E0J1Q4,2.3.1.251,Lipid A palmitoyltransferase PagP (EC 2.3.1.25...,Escherichia coli (strain ATCC 9637 / CCM 2024 ...,Escherichia coli,e_coli,186,558,561,ATGAACGTGAGCAAATATGTGGCGATTTTTAGCTTTGTGTTTATTC...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -1.57\n -...
1,rule0002_148,P0A9S1,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),Escherichia coli,e_coli,382,1146,1149,ATGGCGAACCGCATGATTCTGAACGAAACCGCGTGGTTTGGCCGCG...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -6.78\n -...
2,rule0003_152,A1A9R3,1.1.1.298,Probable malonic semialdehyde reductase RutE (...,Escherichia coli O1:K1 / APEC,Escherichia coli,e_coli,196,588,591,ATGAACGAAGCGGTGAGCCCGGGCGCGCTGAGCACCCTGTTTACCG...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -4.81\n -...
3,rule0003_171,P0A9S1,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),Escherichia coli,e_coli,382,1146,1149,ATGGCGAACCGCATGATTCTGAACGAAACCGCGTGGTTTGGCCGCG...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -6.78\n -...
4,rule0003_173,P06988,1.1.1.23,Histidinol dehydrogenase (HDH) (EC 1.1.1.23),Escherichia coli (strain K12),Escherichia coli,e_coli,434,1302,1305,ATGAGCTTTAACACCATTATTGATTGGAACAGCTGCACCGCGGAAC...,automatic_ecoli_only,Automatically selected highest-scoring Escheri...,===> SUCCESS - all constraints evaluations pas...,===> TOTAL OBJECTIVES SCORE: -12.83\n -1...


### Merge optimized E. coli DNA back to reactions

In [32]:
optimizedGeneForMergeDF = optimizedEcoliGeneDF[
    [
        "ruleName",
        "uniprotAccession",
        "optimizedCds",
        "optimizedCdsLengthBp",
        "expressionHost",
        "targetSpecies",
    ]
].copy()

reactionWithEcoliDnaDesignDF = reactionWithSelectedEnzymeDF.merge(
    optimizedGeneForMergeDF,
    on=["ruleName", "uniprotAccession"],
    how="left"
)

reactionWithEcoliDnaDesignDF["hasOptimizedDna"] = (
    reactionWithEcoliDnaDesignDF["optimizedCds"]
    .fillna("")
    .astype(str)
    .str.len() > 0
)

reactionWithEcoliDnaDesignDF.to_csv(
    os.path.join(dnaResultsDir, "reactionWithEcoliDnaDesignDF.csv"),
    index=False
)

print(reactionWithEcoliDnaDesignDF["hasOptimizedDna"].value_counts(dropna=False))
reactionWithEcoliDnaDesignDF

hasOptimizedDna
True    82040
Name: count, dtype: int64


,reactants,products,reactionString,ruleName,thermo,reactantStoich,productStoich,reactionType,numReactantMolecules,numProductMolecules,...,proteinSequence,selectionReason,enzymeConfidence,selectionMode,hasSelectedEnzyme,optimizedCds,optimizedCdsLengthBp,expressionHost,targetSpecies,hasOptimizedDna
0,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,O=c1[nH]c(C(O)O)nc2c1ncn2[C@@H]1O[C@@H]2O[C@H]...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@]2(C(O)O)O[C@H]2[...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,MAVTKLVLVRHGESQWNKENRFTGWYDVDLSEKGVSEAKAAGKLLK...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGGCGGTGACCAAACTGGTGCTGGTGCGCCATGGCGAAAGCCAGT...,753,Escherichia coli,e_coli,True
1,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,CC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C@...,CC(=O)O[C@H]1[C@H](n2cnc3c(=O)[nH]cnc32)O[C@H]...,rule0001_87,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,MNVSKYVAIFSFVFIQLISVGKVFANADEWMTTFRENIAQTRQQPE...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAACGTGAGCAAATATGTGGCGATTTTTAGCTTTGTGTTTATTC...,561,Escherichia coli,e_coli,True
2,CC(O)(C#N)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH...,CC(O)(C#N)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH...,CC(O)(C#N)O[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH...,rule0165_2,No_Thermo,"(1, 1, 1, 1)","(1, 1, 1)",Enzymatic,4,3,...,MSFNTIIDWNSCTAEQQRQLLMRPAISASESITRTVNDILDNVKAR...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAGCTTTAACACCATTATTGATTGGAACAGCTGCACCGCGGAAC...,1305,Escherichia coli,e_coli,True
3,CC(=O)O[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n...,CC(=O)O[C@@H]1[C@H](OC=O)[C@@H](CO)O[C@H]1n1cn...,CC(=O)O[C@@H]1[C@H](OC(C)=O)[C@@H](CO)O[C@H]1n...,rule0126_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,MSTPLQGIKVLDFTGVQSGPSCTQMLAWFGADVIKIERPGVGDVTR...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAGCACCCCGCTGCAAGGCATTAAAGTGCTGGATTTTACCGGCG...,1251,Escherichia coli,e_coli,True
4,CC(=O)[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,CC(N)[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc32...,CC(=O)[C@H]1[C@@H](O)[C@H](n2cnc3c(=O)[nH]cnc3...,rule0091_14,No_Thermo,"(1, 1, 1)","(1, 1, 1)",Enzymatic,3,3,...,MGSPSLYSARKTTLALAVALSFAWQAPVFAHGGEAHMVPMDKTLKE...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGGGCAGCCCGAGCCTGTATAGCGCGCGCAAAACCACCCTGGCGC...,2274,Escherichia coli,e_coli,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82035,CCC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C...,CCC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C...,CCC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C...,rule0126_2,No_Thermo,"(1, 1)","(1, 1)",Enzymatic,2,2,...,MSTPLQGIKVLDFTGVQSGPSCTQMLAWFGADVIKIERPGVGDVTR...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGAGCACCCCGCTGCAAGGCATTAAAGTGCTGGATTTTACCGGCG...,1251,Escherichia coli,e_coli,True
82036,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@H]2OCC(O...,O=c1[nH]cnc2c1ncn2[C@@H]1OC[C@]2(CO)OCC(O)O[C@...,O=c1[nH]cnc2c1ncn2[C@@H]1O[C@H](CO)[C@H]2OCC(O...,rule0028_50,No_Thermo,"(1,)","(1,)",Enzymatic,1,1,...,MAVTKLVLVRHGESQWNKENRFTGWYDVDLSEKGVSEAKAAGKLLK...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGGCGGTGACCAAACTGGTGCTGGTGCGCCATGGCGAAAGCCAGT...,753,Escherichia coli,e_coli,True
82037,CCC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C...,O=C(O)CCC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cn...,CCC(=O)OC[C@H]1O[C@@H](n2cnc3c(=O)[nH]cnc32)[C...,rule0023_13,No_Thermo,"(1, 1)","(1,)",Enzymatic,2,1,...,MIRTMLQGKLHRVKVTHADLHYEGSCAIDQDFLDAAGILENEAIDI...,Automatically selected highest-scoring Escheri...,automatic_needs_later_review,automatic_ecoli_only,True,ATGATTCGCACCATGCTGCAAGGCAAACTGCATCGCGTGAAAGTGA...,381,Escherichia coli,e_coli,True
82038,COC(=O

### 10. Export E. coli design files for BaseBuddy / Teselagen / j5

In [33]:
ecoliPartImportDF = optimizedEcoliGeneDF.copy()

ecoliPartImportDF["partName"] = (
    ecoliPartImportDF["ruleName"].astype(str)
    + "__"
    + ecoliPartImportDF["uniprotAccession"].astype(str)
    + "__EcoliHost"
)

ecoliPartImportDF["partType"] = "CDS"
ecoliPartImportDF["sequenceType"] = "DNA"
ecoliPartImportDF["assemblyMethod"] = "GoldenGate_or_Gibson"

ecoliPartImportDF["description"] = (
    "DORAnet rule: "
    + ecoliPartImportDF["ruleName"].astype(str)
    + "; EC: "
    + ecoliPartImportDF["ecNumber"].astype(str)
    + "; protein: "
    + ecoliPartImportDF["proteinName"].astype(str)
    + "; source organism: "
    + ecoliPartImportDF["sourceOrganism"].astype(str)
    + "; expression host: Escherichia coli"
)

ecoliPartImportCols = [
    "partName",
    "partType",
    "sequenceType",
    "optimizedCds",
    "description",
    "assemblyMethod",
    "ruleName",
    "uniprotAccession",
    "ecNumber",
    "proteinName",
    "sourceOrganism",
    "expressionHost",
    "targetSpecies",
    "optimizedCdsLengthBp",
]

ecoliPartImportDF[ecoliPartImportCols].to_csv(
    os.path.join(handoffResultsDir, "teselagenEcoliPartImportDF.csv"),
    index=False
)

with open(os.path.join(handoffResultsDir, "ecoliPathwayParts.fasta"), "w", encoding="utf-8") as f:
    for _, row in ecoliPartImportDF.iterrows():
        f.write(f">{row['partName']}\n")
        f.write(str(row["optimizedCds"]) + "\n\n")

print("Saved:")
print(os.path.join(handoffResultsDir, "teselagenEcoliPartImportDF.csv"))
print(os.path.join(handoffResultsDir, "ecoliPathwayParts.fasta"))

ecoliPartImportDF[ecoliPartImportCols].head()

Saved:
/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/07_webtool_handoff/teselagenEcoliPartImportDF.csv
/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/Results/combinedEbolaVirus_bestMACAW_allDB_generative/DNA_design/07_webtool_handoff/ecoliPathwayParts.fasta


,partName,partType,sequenceType,optimizedCds,description,assemblyMethod,ruleName,uniprotAccession,ecNumber,proteinName,sourceOrganism,expressionHost,targetSpecies,optimizedCdsLengthBp
0,rule0001_87__E0J1Q4__EcoliHost,CDS,DNA,ATGAACGTGAGCAAATATGTGGCGATTTTTAGCTTTGTGTTTATTC...,DORAnet rule: rule0001_87; EC: 2.3.1.251; prot...,GoldenGate_or_Gibson,rule0001_87,E0J1Q4,2.3.1.251,Lipid A palmitoyltransferase PagP (EC 2.3.1.25...,Escherichia coli (strain ATCC 9637 / CCM 2024 ...,Escherichia coli,e_coli,561
1,rule0002_148__P0A9S1__EcoliHost,CDS,DNA,ATGGCGAACCGCATGATTCTGAACGAAACCGCGTGGTTTGGCCGCG...,DORAnet rule: rule0002_148; EC: 1.1.1.77; prot...,GoldenGate_or_Gibson,rule0002_148,P0A9S1,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),Escherichia coli,e_coli,1149
2,rule0003_152__A1A9R3__EcoliHost,CDS,DNA,ATGAACGAAGCGGTGAGCCCGGGCGCGCTGAGCACCCTGTTTACCG...,DORAnet rule: rule0003_152; EC: 1.1.1.298; pro...,GoldenGate_or_Gibson,rule0003_152,A1A9R3,1.1.1.298,Probable malonic semialdehyde reductase RutE (...,Escherichia coli O1:K1 / APEC,Escherichia coli,e_coli,591
3,rule0003_171__P0A9S1__EcoliHost,CDS,DNA,ATGGCGAACCGCATGATTCTGAACGAAACCGCGTGGTTTGGCCGCG...,DORAnet rule: rule0003_171; EC: 1.1.1.77; prot...,GoldenGate_or_Gibson,rule0003_171,P0A9S1,1.1.1.77,Lactaldehyde reductase (EC 1.1.1.77) (Propaned...,Escherichia coli (strain K12),Escherichia coli,e_coli,1149
4,rule0003_173__P06988__EcoliHost,CDS,DNA,ATGAGCTTTAACACCATTATTGATTGGAACAGCTGCACCGCGGAAC...,DORAnet rule: rule0003_173; EC: 1.1.1.23; prot...,GoldenGate_or_Gibson,rule0003_173,P06988,1.1.1.23,Histidinol dehydrogenase (HDH) (EC 1.1.1.23),Escherichia coli (strain K12),Escherichia coli,e_coli,1305


In [34]:
ecoliPipelineSummary = {
    "totalReactionRows": len(reactionWithRuleDF),
    "rulesWithRuleLookup": reactionWithRuleDF["hasRuleLookup"].sum(),
    "uniqueRulesWithRuleLookup": reactionWithRuleDF.loc[
        reactionWithRuleDF["hasRuleLookup"], "ruleName"
    ].nunique(),
    "rulesWithSelectedEcoliEnzyme": selectedEcoliEnzymeDF["ruleName"].nunique(),
    "selectedEcoliEnzymes": len(selectedEcoliEnzymeDF),
    "selectedProteinsWithSequence": selectedProteinSequenceDF["hasProteinSequence"].sum(),
    "optimizedEcoliGenes": len(optimizedEcoliGeneDF),
    "reactionRowsWithOptimizedDna": reactionWithEcoliDnaDesignDF["hasOptimizedDna"].sum(),
}

ecoliPipelineSummaryDF = pd.DataFrame(
    list(ecoliPipelineSummary.items()),
    columns=["metric", "value"]
)


ecoliPipelineSummaryDF

,metric,value
0,totalReactionRows,176400
1,rulesWithRuleLookup,176400
2,uniqueRulesWithRuleLookup,168
3,rulesWithSelectedEcoliEnzyme,68
4,selectedEcoliEnzymes,68
5,selectedProteinsWithSequence,68
6,optimizedEcoliGenes,68
7,reactionRowsWithOptimizedDna,82040
